# 6. Data quality issues and decisions

This is a decision log, not an instruction to blindly alter raw records. Review missingness, duplicates, constants, and cardinality; cleaning choices are applied only to a separate analysis copy.

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(project_root))

from src.data_loader import load_data
from src.cleaning import clean_data

df = load_data()
quality = df.isna().sum().rename("missing_count").to_frame()
quality["missing_share"] = df.isna().mean()
quality["distinct_including_missing"] = df.nunique(dropna=False)
quality["constant"] = quality["distinct_including_missing"].eq(1)
display(quality.sort_values("missing_count", ascending=False))
print("Exact duplicates:", df.duplicated().sum())
print("Potential redundant fields: survived/alive, pclass/class, embarked/embark_town")

,missing_count,missing_share,distinct_including_missing,constant
deck,688,0.772166,8,False
age,177,0.198653,89,False
embarked,2,0.002245,4,False
embark_town,2,0.002245,4,False
sex,0,0.000000,2,False
pclass,0,0.000000,3,False
survived,0,0.000000,2,False
fare,0,0.000000,248,False
parch,0,0.000000,7,False
sibsp,0,0.000000,7,False


Exact duplicates: 107
Potential redundant fields: survived/alive, pclass/class, embarked/embark_town


| Issue | Column(s) | Severity | Decision |
|---|---|---|---|
| Missing ages | `age` | Medium | Median impute in analysis copy; retain raw missingness for audit. |
| Missing embarkation | `embarked`, `embark_town` | Low | Mode impute only where missing. |
| Largely missing cabin information | `deck` | High | Preserve a visible `Unknown` category; do not infer a deck. |
| Exact duplicate feature rows | all | Review | Report count; do not drop without a passenger key. |
| Redundant encodings | `survived`/`alive`, `pclass`/`class`, `embarked`/`embark_town` | Medium | Avoid counting redundant fields as independent evidence or model features. |
| Fare and family size are skewed | `fare`, `sibsp`, `parch` | Review | Flag IQR outliers; verify records rather than automatically removing valid extremes. |
| No timestamp or passenger identifier | dataset | Limitation | No temporal trend or person-level deduplication is claimed. |

Never apply SMOTE or feature-selection remedies during EDA; those are modeling decisions that require training-only validation.

In [2]:
cleaned = clean_data(df)
print("Raw rows:", len(df), "| cleaned rows:", len(cleaned))
print("Missing values after documented cleaning:")
display(cleaned.isna().sum().sort_values(ascending=False).head(10))

Raw rows: 891 | cleaned rows: 891
Missing values after documented cleaning:


survived    0
pclass      0
sex         0
age         0
sibsp       0
parch       0
fare        0
embarked    0
class       0
who         0
dtype: int64